# 07 · Reading a plate by redrawing it

The YOLO26s reader is kept as it is. After it, the 5 best valid readings of a plate are drawn as clean plates,
fitted to the photo (corners, brightness, blur), and the reading whose drawing fits best is chosen.

**Inputs:** the `bangla-plate-reading` dataset and the YOLO26s reader `best.pt`.

In [ ]:
%pip install -q ultralytics==8.4.46

In [ ]:
import json
from pathlib import Path

import pandas as pd
import torch
from ultralytics import YOLO

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

DATA = next(p for p in INPUT.rglob("bangla-plate-reading") if p.is_dir())
WEIGHTS = list(INPUT.rglob("best.pt"))
print(DATA)
print(WEIGHTS)

In [ ]:
classes = json.loads((DATA / "classes.json").read_text(encoding="utf-8"))
bangla = [c["bangla"] for c in classes]
kind = [c["kind"] for c in classes]

test_images = sorted((DATA / "images" / "test").iterdir())
print(len(classes), "classes")
print(len(test_images), "test plates")

## Plate text

Same as notebooks 05 and 06: district, word and letter from left to right, then the six digits as `xx-xxxx`.
The true text of a test plate comes from its label file.

In [ ]:
def plate_text(chars):
    # chars: list of (class id, x centre)
    chars = sorted(chars, key=lambda c: c[1])
    head = " ".join(bangla[c] for c, x in chars if kind[c] != "digit")
    digits = "".join(bangla[c] for c, x in chars if kind[c] == "digit")
    if len(digits) == 6:
        digits = digits[:2] + "-" + digits[2:]
    return " ".join(part for part in [head, digits] if part)


def edit_distance(a, b):
    row = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        prev, row[0] = row[0], i
        for j, cb in enumerate(b, 1):
            prev, row[j] = row[j], min(row[j] + 1, row[j - 1] + 1, prev + (ca != cb))
    return row[-1]


def read_labels(image_path, split="test"):
    # YOLO label file -> list of (class id, x centre, y centre, width, height)
    label = DATA / "labels" / split / (image_path.stem + ".txt")
    boxes = []
    for line in label.read_text().splitlines():
        p = line.split()
        if p:
            boxes.append((int(p[0]), *map(float, p[1:])))
    return boxes


def read_truth(image_path, split="test"):
    return plate_text([(b[0], b[1]) for b in read_labels(image_path, split)])


for path in test_images[:3]:
    print(path.name[:50], "->", read_truth(path))
print(edit_distance("ঢাকা মেট্রো গ ২৪-২৯৫৫", "ঢাকা মেট্রো ২৪-২৯৫৫"))

## Baseline: current reader on the test set

The reader at 0.25 confidence, characters sorted left to right, as in the deployed AI service.
This must match notebook 05: 77.4% on all 939 plates (75.7% bd-lpr, 77.9% blpd).

In [ ]:
reader = YOLO(WEIGHTS[0])
device = 0 if torch.cuda.is_available() else "cpu"

rows = []
for start in range(0, len(test_images), 64):
    batch = test_images[start:start + 64]
    results = reader.predict(batch, imgsz=320, conf=0.25, device=device, verbose=False)
    for path, r in zip(batch, results):
        chars = list(zip(r.boxes.cls.int().tolist(), r.boxes.xywhn[:, 0].tolist()))
        truth = read_truth(path)
        pred = plate_text(chars)
        rows.append({"image": path.name, "source": path.name.split("__")[0],
                     "truth": truth, "predicted": pred, "correct": truth == pred,
                     "errors": edit_distance(truth, pred), "length": len(truth)})

baseline = pd.DataFrame(rows)
baseline.head()

In [ ]:
def summary(df):
    return pd.Series({"plates": len(df),
                      "plate acc %": round(100 * df["correct"].mean(), 1),
                      "CER %": round(100 * df["errors"].sum() / df["length"].sum(), 2)})


table = baseline.groupby("source").apply(summary)
table.loc["all"] = summary(baseline)
table

## Plate rules: the 5 best valid readings

A Bangladeshi plate has one district, at most one word (Metro / Chatto / Raj), one series letter and six digits.

The reader is run again at 0.05 confidence so every character position keeps its weaker guesses too.
Guesses that overlap are one position. Each position either takes one of its (up to 3) guesses, scored `log(conf)`,
or is skipped as noise, scored `log(1 - best conf)`. A beam search keeps only readings that follow the plate pattern,
and the 5 best go to the redrawing step.

In [ ]:
def reader_guesses(images, conf=0.05):
    out = {}
    for start in range(0, len(images), 64):
        batch = images[start:start + 64]
        results = reader.predict(batch, imgsz=320, conf=conf, device=device, max_det=100, verbose=False)
        for path, r in zip(batch, results):
            out[path.name] = [
                {"cls": c, "conf": p, "x": xywh[0], "box": xyxy}
                for c, p, xywh, xyxy in zip(r.boxes.cls.int().tolist(), r.boxes.conf.tolist(),
                                            r.boxes.xywhn.tolist(), r.boxes.xyxyn.tolist())
            ]
    return out


test_guesses = reader_guesses(test_images)
print(sum(len(g) for g in test_guesses.values()), "guesses on", len(test_guesses), "plates")

In [ ]:
import math

LIMIT = {"digit": 6, "district": 1, "letter": 1, "word": 1}
KINDS = list(LIMIT)


def iou(a, b):
    w = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    h = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = w * h
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union else 0.0


def positions(guesses):
    # overlapping guesses are one character position, best guess first
    groups = []
    for g in sorted(guesses, key=lambda g: -g["conf"]):
        for group in groups:
            if iou(group[0]["box"], g["box"]) > 0.5:
                group.append(g)
                break
        else:
            groups.append([g])
    return sorted(groups, key=lambda group: group[0]["x"])


def top_readings(guesses, k=5, beam=256):
    groups = positions(guesses)
    # a state is (score, counts per kind, chosen guesses)
    states = [(0.0, (0, 0, 0, 0), ())]
    for i, group in enumerate(groups):
        left = len(groups) - i - 1
        options = group[:3]
        new_states = []
        for score, counts, chosen in states:
            # skip this position
            new_states.append((score + math.log(max(1e-4, 1 - options[0]["conf"])), counts, chosen))
            # or take one of its guesses
            for g in options:
                j = KINDS.index(kind[g["cls"]])
                c = list(counts)
                c[j] += 1
                if c[j] <= LIMIT[KINDS[j]]:
                    new_states.append((score + math.log(max(1e-4, g["conf"])), tuple(c), chosen + (g,)))
        # drop states that can no longer reach six digits, a district and a letter
        new_states = [s for s in new_states if (6 - s[1][0]) + (1 - s[1][1]) + (1 - s[1][2]) <= left]
        states = sorted(new_states, key=lambda s: -s[0])[:beam]

    readings = {}
    for score, counts, chosen in states:
        if counts[:3] == (6, 1, 1):
            text = plate_text([(g["cls"], g["x"]) for g in chosen])
            if text not in readings or score > readings[text]["logp"]:
                readings[text] = {"text": text, "logp": score, "chars": [(g["cls"], g["box"]) for g in chosen]}
    return sorted(readings.values(), key=lambda r: -r["logp"])[:k]


example = test_images[0].name
for r in top_readings(test_guesses[example]):
    print(round(r["logp"], 2), r["text"])
print("truth:", read_truth(test_images[0]))

In [ ]:
rules = []
for path in test_images:
    readings = top_readings(test_guesses[path.name])
    truth = read_truth(path)
    pred = readings[0]["text"] if readings else ""
    rules.append({"image": path.name, "source": path.name.split("__")[0], "truth": truth,
                  "predicted": pred, "correct": truth == pred,
                  "errors": edit_distance(truth, pred), "length": len(truth),
                  "in_top5": any(r["text"] == truth for r in readings), "n_readings": len(readings)})
rules = pd.DataFrame(rules)

compare = pd.DataFrame({
    "baseline acc %": baseline.groupby("source")["correct"].mean() * 100,
    "plate rules acc %": rules.groupby("source")["correct"].mean() * 100,
    "truth in top 5 %": rules.groupby("source")["in_top5"].mean() * 100,
})
compare.loc["all"] = [baseline["correct"].mean() * 100, rules["correct"].mean() * 100, rules["in_top5"].mean() * 100]
print("plates with no valid reading:", (rules["n_readings"] == 0).sum())
compare.round(1)

## Glyph atlas

To draw a plate we need one clean image per character class. For each class we cut its boxes out of the training crops,
keep the 40 tallest (the sharpest), and take their pixel-wise median. No network is trained here.

Every crop is first turned grey, its contrast stretched to 0–1, and flipped if needed so the ink is dark on a light plate.

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

PH, PW = 80, 160   # every plate is compared at this size
GLYPH = 48         # size of one atlas image
PAD = 0.08         # margin around a character box, the same when cutting and drawing


def normalise(gray):
    lo, hi = np.percentile(gray, [2, 98])
    g = np.clip((gray - lo) / max(hi - lo, 1e-3), 0, 1).astype(np.float32)
    if np.median(g) < 0.5:   # light ink on a dark plate
        g = 1 - g
    return g


def load_plate(path):
    # returns the normalised crop at full size and at the compare size
    full = normalise(np.asarray(Image.open(path).convert("L"), np.float32) / 255)
    small = Image.fromarray((full * 255).astype(np.uint8)).resize((PW, PH), Image.BILINEAR)
    return full, np.asarray(small, np.float32) / 255


def to_corners(box):
    # YOLO (x centre, y centre, w, h) -> (x0, y0, x1, y1), all relative to the crop
    x, y, w, h = box
    return (x - w / 2, y - h / 2, x + w / 2, y + h / 2)


def padded(box):
    x0, y0, x1, y1 = box
    dx, dy = (x1 - x0) * PAD, (y1 - y0) * PAD
    return (x0 - dx, y0 - dy, x1 + dx, y1 + dy)


def resize(img, w, h):
    return np.asarray(Image.fromarray((img * 255).astype(np.uint8)).resize((w, h), Image.BILINEAR), np.float32) / 255

In [ ]:
from collections import defaultdict

train_images = sorted((DATA / "images" / "train").iterdir())
pool = defaultdict(list)   # class -> [(height in pixels, glyph image)]

for path in train_images:
    full, _ = load_plate(path)
    h, w = full.shape
    for cls, *box in read_labels(path, "train"):
        if len(pool[cls]) >= 400:
            continue
        x0, y0, x1, y1 = padded(to_corners(box))
        x0, x1 = int(max(0, x0 * w)), int(min(w, x1 * w))
        y0, y1 = int(max(0, y0 * h)), int(min(h, y1 * h))
        if x1 - x0 >= 3 and y1 - y0 >= 3:
            pool[cls].append((y1 - y0, resize(full[y0:y1, x0:x1], GLYPH, GLYPH)))

atlas = {}
for cls, items in pool.items():
    items.sort(key=lambda t: -t[0])
    atlas[cls] = np.median(np.stack([g for _, g in items[:40]]), axis=0)

missing = [classes[c]["name"] for c in range(len(classes)) if c not in atlas]
print(len(train_images), "training crops")
print(len(atlas), "of", len(classes), "classes in the atlas, missing:", missing or "none")
np.savez_compressed(WORK / "glyph_atlas.npz", **{str(c): g for c, g in atlas.items()})

In [ ]:
cols = 16
rows = math.ceil(len(atlas) / cols)
fig, axes = plt.subplots(rows, cols, figsize=(cols * 0.9, rows))
for ax in axes.flat:
    ax.axis("off")
for ax, cls in zip(axes.flat, sorted(atlas)):
    ax.imshow(atlas[cls], cmap="gray", vmin=0, vmax=1)
    ax.set_title(classes[cls]["name"][:8], fontsize=6)
plt.tight_layout()
plt.savefig(WORK / "atlas.png", dpi=130)
plt.show()

## Drawing a plate

A reading is drawn on a white plate: each character's atlas image is pasted into its box, and a dash is drawn between the
second and third digit. Below, test plates drawn from their **true** labels next to the photo, to check the atlas looks right.

In [ ]:
def draw_plate(chars):
    # chars: list of (class id, (x0, y0, x1, y1)) relative to the crop
    plate = np.ones((PH, PW), np.float32)
    for cls, box in chars:
        if cls not in atlas:
            continue
        x0, y0, x1, y1 = padded(box)
        px0, px1 = round(x0 * PW), round(x1 * PW)
        py0, py1 = round(y0 * PH), round(y1 * PH)
        if px1 - px0 < 2 or py1 - py0 < 2:
            continue
        glyph = resize(atlas[cls], px1 - px0, py1 - py0)
        # clip to the plate edges
        cx0, cy0, cx1, cy1 = max(px0, 0), max(py0, 0), min(px1, PW), min(py1, PH)
        if cx1 > cx0 and cy1 > cy0:
            part = glyph[cy0 - py0:cy1 - py0, cx0 - px0:cx1 - px0]
            plate[cy0:cy1, cx0:cx1] = np.minimum(plate[cy0:cy1, cx0:cx1], part)

    digits = sorted([box for cls, box in chars if kind[cls] == "digit"])
    if len(digits) == 6 and digits[2][0] > digits[1][2]:
        left, right = digits[1][2], digits[2][0]
        yc = np.mean([(b[1] + b[3]) / 2 for b in digits])
        dh = np.mean([b[3] - b[1] for b in digits])
        x0, x1 = round((left + 0.2 * (right - left)) * PW), round((right - 0.2 * (right - left)) * PW)
        y0 = round((yc - 0.06 * dh) * PH)
        plate[max(y0, 0):max(y0 + 1, round((yc + 0.06 * dh) * PH)), max(x0, 0):x1] = 0.0
    return plate


fig, axes = plt.subplots(6, 2, figsize=(6, 9))
for row, path in zip(axes, test_images[::len(test_images) // 6]):
    _, photo = load_plate(path)
    truth_chars = [(cls, to_corners(box)) for cls, *box in read_labels(path)]
    row[0].imshow(photo, cmap="gray", vmin=0, vmax=1)
    row[0].set_title("photo", fontsize=8)
    row[1].imshow(draw_plate(truth_chars), cmap="gray", vmin=0, vmax=1)
    row[1].set_title("drawn", fontsize=8)
    for ax in row:
        ax.axis("off")
plt.tight_layout()
plt.savefig(WORK / "drawn_vs_photo.png", dpi=130)
plt.show()

## Fitting a drawing to the photo

A drawing is moved onto the photo with three steps, all written in PyTorch so they can be adjusted by gradient descent:

1. **Warp:** the 4 corners of the drawing are moved (a perspective transform), which handles tilt.
2. **Blur:** a Gaussian blur with its own sigma, which handles out-of-focus and motion blur.
3. **Light:** a gain and a bias on the grey values, which handles dark plates.

The drawing is compared with the photo either by **pixels** or by **edges** (Sobel), which ignores dirt and uneven light.
All readings of a plate are fitted together as one batch.

In [ ]:
import torch.nn.functional as F

device_t = "cuda" if torch.cuda.is_available() else "cpu"


def homography(src, dst):
    # (B, 4, 2) point pairs -> (B, 3, 3) matrices that map src to dst
    rows, rhs = [], []
    for i in range(4):
        x, y = src[:, i, 0], src[:, i, 1]
        u, v = dst[:, i, 0], dst[:, i, 1]
        one, zero = torch.ones_like(x), torch.zeros_like(x)
        rows.append(torch.stack([x, y, one, zero, zero, zero, -u * x, -u * y], -1))
        rows.append(torch.stack([zero, zero, zero, x, y, one, -v * x, -v * y], -1))
        rhs += [u, v]
    h = torch.linalg.solve(torch.stack(rows, 1), torch.stack(rhs, 1))
    return torch.cat([h, torch.ones_like(h[:, :1])], 1).view(-1, 3, 3)


def warp(img, offsets):
    # move the 4 corners of each image by offsets (B, 4, 2), in [-1, 1] units
    b, _, h, w = img.shape
    corners = torch.tensor([[-1., -1.], [1., -1.], [1., 1.], [-1., 1.]], device=img.device).expand(b, 4, 2)
    m = homography(corners + offsets, corners)   # output pixel -> drawing pixel
    ys = torch.linspace(-1 + 1 / h, 1 - 1 / h, h, device=img.device)
    xs = torch.linspace(-1 + 1 / w, 1 - 1 / w, w, device=img.device)
    gy, gx = torch.meshgrid(ys, xs, indexing="ij")
    pts = torch.stack([gx, gy, torch.ones_like(gx)], -1).view(1, -1, 3).expand(b, -1, 3)
    src = pts @ m.transpose(1, 2)
    grid = (src[..., :2] / src[..., 2:].clamp(min=1e-6)).view(b, h, w, 2)
    return F.grid_sample(img, grid, padding_mode="border", align_corners=False)


def blur(img, sigma):
    # Gaussian blur with one sigma per image
    b, _, h, w = img.shape
    r = torch.arange(-4, 5, device=img.device, dtype=img.dtype)
    k = torch.exp(-(r.view(1, -1) ** 2) / (2 * sigma.view(-1, 1) ** 2))
    k = k / k.sum(1, keepdim=True)
    x = img.view(1, b, h, w)
    x = F.conv2d(F.pad(x, (4, 4, 0, 0), mode="replicate"), k.view(b, 1, 1, 9), groups=b)
    x = F.conv2d(F.pad(x, (0, 0, 4, 4), mode="replicate"), k.view(b, 1, 9, 1), groups=b)
    return x.view(b, 1, h, w)


SOBEL = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]]).view(1, 1, 3, 3)


def edges(img):
    s = SOBEL.to(img.device)
    p = F.pad(img, (1, 1, 1, 1), mode="replicate")
    e = torch.sqrt(F.conv2d(p, s) ** 2 + F.conv2d(p, s.transpose(2, 3)) ** 2 + 1e-6)
    return e / (e.mean((1, 2, 3), keepdim=True) + 1e-6)


def match_light(x, photo):
    # best gain and bias in closed form, so x has the photo's grey values
    xm, pm = x.mean((1, 2, 3), keepdim=True), photo.mean((1, 2, 3), keepdim=True)
    a = ((x - xm) * (photo - pm)).mean((1, 2, 3), keepdim=True) / (((x - xm) ** 2).mean((1, 2, 3), keepdim=True) + 1e-6)
    return a * x + (pm - a * xm)

In [ ]:
STEPS = 40
REG = 0.5   # keeps the corners close to where the reader put the characters


def fit(drawings, photo, mode="edge", steps=STEPS):
    # drawings (k, 1, H, W), photo (1, 1, H, W) -> fitted drawings and one distance per drawing
    k = len(drawings)
    target = photo.expand(k, -1, -1, -1)
    if mode == "edge":
        target = edges(target)

    offsets = torch.zeros(k, 4, 2, device=drawings.device, requires_grad=True)
    gain = torch.ones(k, 1, 1, 1, device=drawings.device, requires_grad=True)
    bias = torch.zeros(k, 1, 1, 1, device=drawings.device, requires_grad=True)
    s = torch.full((k,), -0.5, device=drawings.device, requires_grad=True)   # sigma = softplus(s) + 0.3
    opt = torch.optim.Adam([{"params": [offsets], "lr": 0.01}, {"params": [gain, bias, s], "lr": 0.05}])

    for _ in range(steps):
        out = gain * blur(warp(drawings, offsets), F.softplus(s) + 0.3) + bias
        diff = (edges(out) if mode == "edge" else out) - target
        loss = diff.abs().mean((1, 2, 3)).sum() + REG * (offsets ** 2).sum()
        opt.zero_grad()
        loss.backward()
        opt.step()

    with torch.no_grad():
        out = match_light(blur(warp(drawings, offsets), F.softplus(s) + 0.3), photo.expand(k, -1, -1, -1))
        diff = (edges(out) if mode == "edge" else out) - target
        dist = diff.abs().mean((1, 2, 3))
    return out, dist

### One plate

A test plate where the plate rules picked the wrong reading but the right one is among its 5 readings.
Each reading is drawn, fitted, and its distance to the photo printed. The right reading should have the smallest distance.

In [ ]:
def fit_plate(path, mode="edge"):
    readings = top_readings(test_guesses[path.name])
    drawings = torch.from_numpy(np.stack([draw_plate(r["chars"]) for r in readings])).unsqueeze(1).to(device_t)
    _, photo = load_plate(path)
    photo = torch.from_numpy(photo).view(1, 1, PH, PW).to(device_t)
    fitted, dist = fit(drawings, photo, mode)
    return readings, photo, fitted, dist


wrong_but_close = rules[(~rules["correct"]) & rules["in_top5"]]
print(len(wrong_but_close), "test plates: rules wrong, truth in top 5")

path = DATA / "images" / "test" / wrong_but_close.iloc[0]["image"]
truth = read_truth(path)
for mode in ["pixel", "edge"]:
    readings, photo, fitted, dist = fit_plate(path, mode)
    print(f"\n{mode}")
    for r, d in zip(readings, dist.tolist()):
        print(f"  {d:.3f}  logp {r['logp']:6.2f}  {r['text']}", "<- truth" if r["text"] == truth else "")

fig, axes = plt.subplots(1, len(readings) + 1, figsize=(2.4 * (len(readings) + 1), 1.6))
axes[0].imshow(photo[0, 0].cpu(), cmap="gray", vmin=0, vmax=1)
axes[0].set_title("photo", fontsize=8)
for ax, img, d, r in zip(axes[1:], fitted, dist.tolist(), readings):
    ax.imshow(img[0].cpu(), cmap="gray")
    ax.set_title(f"{d:.3f}" + (" truth" if r["text"] == truth else ""), fontsize=8)
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## Fit every validation and test plate

The validation plates are used only to choose the weight between the reader's score and the fit distance.
For each plate, all its readings are drawn and fitted twice: with pixels and with edges.

In [ ]:
import time

val_images = sorted((DATA / "images" / "val").iterdir())
val_guesses = reader_guesses(val_images)


def fit_split(images, guesses, keep_fitted=False):
    plates = []
    for path in images:
        readings = top_readings(guesses[path.name])
        plate = {"image": path.name, "path": path, "readings": readings}
        if readings:
            drawings = torch.from_numpy(np.stack([draw_plate(r["chars"]) for r in readings])).unsqueeze(1).to(device_t)
            _, photo = load_plate(path)
            photo = torch.from_numpy(photo).view(1, 1, PH, PW).to(device_t)
            _, d_pixel = fit(drawings, photo, "pixel")
            fitted, d_edge = fit(drawings, photo, "edge")
            for r, a, b in zip(readings, d_pixel.tolist(), d_edge.tolist()):
                r["dist"] = {"pixel": a, "edge": b}
            plate["photo"] = photo
            plate["fitted"] = fitted   # edge fit, used by the next two comparisons
        plates.append(plate)
    return plates


t0 = time.time()
val_plates = fit_split(val_images, val_guesses)
test_plates = fit_split(test_images, test_guesses)
ms = 1000 * (time.time() - t0) / (len(val_plates) + len(test_plates))
print(len(val_plates), "val plates,", len(test_plates), "test plates")
print(f"{ms:.1f} ms per plate to draw and fit its readings twice")

## Comparing with the reader's own features

A third way to compare: pass the fitted drawing and the photo through the first layers of the YOLO26s reader
(up to stride 8) and take the cosine distance. No training.

In [ ]:
backbone = YOLO(WEIGHTS[0]).model.to(device_t).float().eval()
FEAT_LAYER = 4


def reader_features(x):
    x = F.interpolate(x.repeat(1, 3, 1, 1), size=(96, 192), mode="bilinear", align_corners=False)
    for i, layer in enumerate(backbone.model):
        if layer.f != -1 or i > FEAT_LAYER:
            break
        x = layer(x)
    return F.normalize(x.flatten(1), dim=1)


with torch.no_grad():
    for plate in val_plates + test_plates:
        if plate["readings"]:
            d = 1 - (reader_features(plate["fitted"]) @ reader_features(plate["photo"]).T).squeeze(1)
            for r, v in zip(plate["readings"], d.tolist()):
                r["dist"]["feature"] = v
print("done")

## A learned comparison

A small encoder (about 1 M parameters) is trained so that a photo and the drawing of its own text land close together,
and drawings with one or two wrong characters land further away. Training pairs come for free: each training crop and
its label drawn with the atlas. The drawings get a random small warp, blur and light change so they look like photos.

In [ ]:
import random
import torch.nn as nn


class PlateEncoder(nn.Module):
    def __init__(self, dim=128):
        super().__init__()
        def block(i, o):
            return nn.Sequential(nn.Conv2d(i, o, 3, 2, 1), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
                                 nn.Conv2d(o, o, 3, 1, 1), nn.BatchNorm2d(o), nn.ReLU(inplace=True))
        self.net = nn.Sequential(block(1, 32), block(32, 64), block(64, 96), block(96, 128))   # 80x160 -> 5x10
        self.head = nn.Linear(128 * 5 * 10, dim)

    def forward(self, x):
        return F.normalize(self.head(self.net(x).flatten(1)), dim=1)


by_kind = defaultdict(list)
for c, k in enumerate(kind):
    by_kind[k].append(c)


def wrong_reading(chars, rng):
    # same boxes, one or two characters swapped for another class of the same kind
    chars = list(chars)
    for i in rng.sample(range(len(chars)), k=min(len(chars), rng.choice([1, 1, 2]))):
        cls, box = chars[i]
        others = [c for c in by_kind[kind[cls]] if c != cls and c in atlas]
        if others:
            chars[i] = (rng.choice(others), box)
    return chars


def camera_like(drawings):
    b = len(drawings)
    with torch.no_grad():
        x = warp(drawings, torch.randn(b, 4, 2, device=drawings.device) * 0.03)
        x = blur(x, torch.rand(b, device=drawings.device) * 1.2 + 0.3)
        x = x * (0.6 + 0.6 * torch.rand(b, 1, 1, 1, device=drawings.device)) + 0.2 * (torch.rand(b, 1, 1, 1, device=drawings.device) - 0.5)
    return x.clamp(0, 1)


train_photos, train_chars = [], []
for path in train_images:
    chars = [(cls, to_corners(box)) for cls, *box in read_labels(path, "train")]
    if chars:
        train_photos.append((load_plate(path)[1] * 255).astype(np.uint8))
        train_chars.append(chars)
train_photos = np.stack(train_photos)
print(len(train_photos), "training plates for the encoder")

In [ ]:
EPOCHS, BATCH, NEG = 12, 64, 4
rng = random.Random(0)
torch.manual_seed(0)

encoder = PlateEncoder().to(device_t)
opt = torch.optim.AdamW(encoder.parameters(), lr=2e-3, weight_decay=1e-4)
steps = EPOCHS * (len(train_photos) // BATCH)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-3, total_steps=steps)

t0 = time.time()
for epoch in range(EPOCHS):
    order = list(range(len(train_photos)))
    rng.shuffle(order)
    total, right, seen = 0.0, 0, 0
    for s in range(0, len(order) - BATCH + 1, BATCH):
        idx = order[s:s + BATCH]
        drawings = []
        for i in idx:
            drawings.append(draw_plate(train_chars[i]))                                   # own text
            drawings += [draw_plate(wrong_reading(train_chars[i], rng)) for _ in range(NEG)]   # wrong texts
        d = camera_like(torch.from_numpy(np.stack(drawings)).unsqueeze(1).to(device_t))
        p = torch.from_numpy(train_photos[idx]).float().div(255).unsqueeze(1).to(device_t)

        zp = encoder(p)
        zd = encoder(d).view(len(idx), NEG + 1, -1)
        logits = (zd @ zp.unsqueeze(2)).squeeze(2) / 0.1
        loss = F.cross_entropy(logits, torch.zeros(len(idx), dtype=torch.long, device=device_t))
        opt.zero_grad()
        loss.backward()
        opt.step()
        sched.step()

        total += loss.item() * len(idx)
        right += (logits.argmax(1) == 0).sum().item()
        seen += len(idx)
    print(f"epoch {epoch + 1:2d}  loss {total / seen:.3f}  picks its own drawing {100 * right / seen:.1f}%")

encoder.eval()
torch.save(encoder.state_dict(), WORK / "plate_encoder.pt")
print(f"{(time.time() - t0) / 60:.1f} min")

In [ ]:
# check on validation labels: own drawing against four wrong ones (chance is 20%)
rng = random.Random(1)
right = n = 0
with torch.no_grad():
    for path in val_images:
        chars = [(cls, to_corners(box)) for cls, *box in read_labels(path, "val")]
        if not chars:
            continue
        d = np.stack([draw_plate(chars)] + [draw_plate(wrong_reading(chars, rng)) for _ in range(4)])
        zd = encoder(torch.from_numpy(d).unsqueeze(1).to(device_t))
        zp = encoder(torch.from_numpy(load_plate(path)[1]).view(1, 1, PH, PW).to(device_t))
        right += int((zd @ zp.T).argmax().item() == 0)
        n += 1
print(f"validation: picks its own drawing for {100 * right / n:.1f}% of {n} plates")

with torch.no_grad():
    for plate in val_plates + test_plates:
        if plate["readings"]:
            d = 1 - (encoder(plate["fitted"]) @ encoder(plate["photo"]).T).squeeze(1)
            for r, v in zip(plate["readings"], d.tolist()):
                r["dist"]["learned"] = v

## Choosing a reading

`score = reader log-probability − w × (distance − smallest distance on that plate)`

`w = 0` is the plate rules alone. For each comparison, `w` is chosen on **validation** and then used once on test.

In [ ]:
SCORES = ["pixel", "edge", "feature", "learned"]
GRID = [0, 0.5, 1, 2, 3, 5, 8, 12, 20, 30, 50, 80, 120, 200, 400]


def pick(plate, score=None, w=0):
    readings = plate["readings"]
    if not readings:
        return ""
    if score is None or w == 0:
        return readings[0]["text"]
    low = min(r["dist"][score] for r in readings)
    return max(readings, key=lambda r: r["logp"] - w * (r["dist"][score] - low))["text"]


for p in val_plates:
    p["truth"] = read_truth(p["path"], "val")
for p in test_plates:
    p["truth"] = read_truth(p["path"])

best_w = {}
for score in SCORES:
    acc = [np.mean([pick(p, score, w) == p["truth"] for p in val_plates]) for w in GRID]
    best_w[score] = GRID[int(np.argmax(acc))]
    print(f"{score:8s} w = {best_w[score]:>4}   validation {100 * max(acc):.2f}%   (rules alone {100 * acc[0]:.2f}%)")

## Test subsets

Fixed from the plates themselves, with thresholds taken from validation:
**dark** (lowest 20% raw brightness), **tilted** (highest 20% angle of the line through the digit centres), **small** (narrowest 20% of crops).

In [ ]:
def plate_facts(path, split):
    img = Image.open(path)
    w, h = img.size
    brightness = float(np.asarray(img.convert("L"), np.float32).mean() / 255)
    pts = [(b[1] * w, b[2] * h) for b in read_labels(path, split) if kind[b[0]] == "digit"]
    tilt = 0.0
    if len(pts) >= 3:
        x, y = np.array(pts).T
        if np.ptp(x) > 0:
            tilt = abs(math.degrees(math.atan(np.polyfit(x, y, 1)[0])))
    return brightness, tilt, w


val_facts = pd.DataFrame([plate_facts(p, "val") for p in val_images], columns=["brightness", "tilt", "width"])
limits = {"dark": val_facts["brightness"].quantile(0.2),
          "tilted": val_facts["tilt"].quantile(0.8),
          "small": val_facts["width"].quantile(0.2)}
print({k: round(float(v), 3) for k, v in limits.items()})

for p in test_plates:
    b, t, w = plate_facts(p["path"], "test")
    p["subsets"] = ["all"] + [name for name, inside in [("dark", b <= limits["dark"]),
                                                         ("tilted", t >= limits["tilted"]),
                                                         ("small", w <= limits["small"])] if inside]
    p["subsets"].append(p["image"].split("__")[0])

pd.Series([s for p in test_plates for s in p["subsets"]]).value_counts()

## Results on test

Plate accuracy (%) for every method on all test plates, each subset and each source. The last row is the upper limit:
how often the true plate is among the 5 readings.

In [ ]:
base_pred = dict(zip(baseline["image"], baseline["predicted"]))

methods = {"baseline (today)": lambda p: base_pred[p["image"]],
           "plate rules": lambda p: pick(p)}
for score in SCORES:
    methods[f"redraw · {score}"] = (lambda s: lambda p: pick(p, s, best_w[s]))(score)

rows = []
for p in test_plates:
    for name, method in methods.items():
        pred = method(p)
        for subset in p["subsets"]:
            rows.append({"image": p["image"], "subset": subset, "method": name, "truth": p["truth"], "predicted": pred,
                         "correct": pred == p["truth"], "errors": edit_distance(p["truth"], pred), "length": len(p["truth"])})
    for subset in p["subsets"]:
        rows.append({"image": p["image"], "subset": subset, "method": "truth in top 5",
                     "correct": any(r["text"] == p["truth"] for r in p["readings"])})
results = pd.DataFrame(rows)
results.to_csv(WORK / "redraw_test_plates.csv", index=False)

order = list(methods) + ["truth in top 5"]
columns = ["all", "dark", "tilted", "small", "bd-lpr", "blpd"]
table = results.pivot_table(index="method", columns="subset", values="correct", aggfunc="mean").mul(100).round(1)
table = table.reindex(index=order, columns=[c for c in columns if c in table.columns])
counts = results[results["method"] == "plate rules"].groupby("subset").size()
table.columns = [f"{c} ({counts[c]})" for c in table.columns]

all_rows = results[(results["subset"] == "all") & (results["method"] != "truth in top 5")]
table["CER all %"] = all_rows.groupby("method").apply(lambda d: 100 * d["errors"].sum() / d["length"].sum()).round(2)
table.to_csv(WORK / "redraw_results.csv")
table

## What changed

Plates the best redraw method fixed and broke, compared with the plate rules alone.

In [ ]:
all_col = table.columns[0]
best = max(SCORES, key=lambda s: table.loc[f"redraw · {s}", all_col])
print("best comparison:", best)

by_image = results[results["subset"] == "all"].pivot(index="image", columns="method", values="predicted")
truth = results[results["subset"] == "all"].drop_duplicates("image").set_index("image")["truth"]
rules_pred, redraw_pred = by_image["plate rules"], by_image[f"redraw · {best}"]
fixed = by_image.index[(rules_pred != truth) & (redraw_pred == truth)]
broken = by_image.index[(rules_pred == truth) & (redraw_pred != truth)]
print(f"fixed {len(fixed)}   broken {len(broken)}\n")
for title, idx in [("Fixed", fixed), ("Broken", broken)]:
    print(title)
    print(pd.DataFrame({"truth": truth[idx], "rules": rules_pred[idx], "redraw": redraw_pred[idx]}).head(15).to_string(index=False))
    print()

In [ ]:
plates_by_name = {p["image"]: p for p in test_plates}
show = [plates_by_name[i] for i in list(fixed)[:5]]
fig, axes = plt.subplots(len(show), 6, figsize=(13, 1.5 * len(show)), squeeze=False)
for row, p in zip(axes, show):
    row[0].imshow(p["photo"][0, 0].cpu(), cmap="gray", vmin=0, vmax=1)
    row[0].set_title("photo", fontsize=7)
    for ax, img, r in zip(row[1:], p["fitted"], p["readings"]):
        ax.imshow(img[0].cpu(), cmap="gray")
        ax.set_title(f"{best} {r['dist'][best]:.3f}" + (" truth" if r["text"] == p["truth"] else ""), fontsize=7)
    for ax in row:
        ax.axis("off")
plt.tight_layout()
plt.savefig(WORK / "fit_examples.png", dpi=130)
plt.show()

## Save

Settings for the AI service and the demo. The atlas and the encoder were saved above.

In [ ]:
config = {"PH": PH, "PW": PW, "GLYPH": GLYPH, "PAD": PAD, "STEPS": STEPS, "REG": REG, "FEAT_LAYER": FEAT_LAYER,
          "top_k": 5, "guess_conf": 0.05, "best_w": best_w, "best_score": best,
          "subset_limits": {k: float(v) for k, v in limits.items()}}
(WORK / "redraw_config.json").write_text(json.dumps(config, indent=1))
print(json.dumps(config, indent=1))
print(sorted(p.name for p in WORK.iterdir()))